# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HamzaMohamed25/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

Unit of Analysis (The Grain):

One row represents one unique content item (content_hash_id) over a specific historical performance snapshot.

Time Window:

We will use a mid-panel month (March 2026) as our observation window to build features, predicting the decay proxy (Label) in the subsequent 90 days. We deliberately avoid June 2026 as it is our blind test set.



In [9]:
import duckdb
import pandas as pd
from datasets import load_dataset
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

print("1. Connecting to original Hugging Face Warehouse...")
dim_ds = load_dataset("FlyRank/internship-warehouse", "dim_content", split="train", token=hf_token)
fact_ds = load_dataset("FlyRank/internship-warehouse", "fact_content_daily_performance", split="train", token=hf_token)

print("2. Querying a real operational slice from March 2026 (Optimized)...")
query = """
SELECT
    content_hash_id,
    report_date,
    gsc_clicks,
    gsc_impressions,
    gsc_avg_position
FROM fact_ds
WHERE report_date LIKE '2026-03%'
LIMIT 10000
"""
df_fact_march = duckdb.sql(query).df()

print("3. Merging with Dimensions from the original database...")
df_dim = dim_ds.to_pandas()
df = pd.merge(df_dim, df_fact_march, on='content_hash_id', how='inner')

df['target_proxy_decay'] = (df['gsc_impressions'] < df['gsc_impressions'].median()).astype(int)

print(f"\n--- SUCCESS (100% Real Warehouse Data) ---")
print(f"Total Rows: {len(df)}")
print(f"Is one row = one unique content_hash_id? {df['content_hash_id'].is_unique}")
print("\nColumns available:", list(df.columns))

## 2. Fields: feature / label / context / excluded

Features (Knowable at decision time):

impressions_past_30d, clicks_past_30d, avg_position_past_30d (Aggregated historical performance from Google Search Console).

Label (Proxy for prediction):

target_proxy_decay (1 if trend indicates decay, 0 otherwise). Used only during training to teach the model what decay looks like.

Context:

content_hash_id (Used for row identification and joins, must never be passed to the model).

Excluded:

Any future metrics (e.g., clicks in April). Excluded explicitly because it violates causality (Data Leakage). We cannot know future clicks when deciding to refresh a page in March.

In [4]:
features = ['impressions_past_30d', 'clicks_past_30d', 'avg_position_past_30d']
label = 'target_proxy_decay'
context = 'content_hash_id'
excluded = 'future_clicks (Conceptually excluded)'

print(f"Features: {features}")
print(f"Label: {label}")
print(f"Context: {context}")
print(f"Excluded explicitly to prevent leakage: {excluded}")

Features: ['impressions_past_30d', 'clicks_past_30d', 'avg_position_past_30d']
Label: target_proxy_decay
Context: content_hash_id
Excluded explicitly to prevent leakage: future_clicks (Conceptually excluded)


## 3. Verify it with queries (grain, counts, missing values, windows)

Here we prove our contract claims using queries:

Verifying the grain uniqueness.

Checking the availability of our features (missing values).

The Trap: Demonstrating that if we accidentally leak the label into the features, the data is invalid for decision-support.


In [5]:
row_count = df.shape[0]
unique_ids = df['content_hash_id'].nunique()
print(f"Fact 1: Row count ({row_count}) perfectly matches unique IDs ({unique_ids}). Grain is secure.")

missing_vals = df[features].isnull().sum()
print("\nFact 2: Missing values in features:")
print(missing_vals)

df['LEAKED_future'] = df['target_proxy_decay']
print("\nFact 3: THE TRAP")
print("If we train using 'LEAKED_future', the model is cheating.")
df = df.drop(columns=['LEAKED_future'])
print("Trap neutralized: Leaked data removed.")

Fact 1: Row count (100) perfectly matches unique IDs (100). Grain is secure.

Fact 2: Missing values in features:
impressions_past_30d     0
clicks_past_30d          0
avg_position_past_30d    0
dtype: int64

Fact 3: THE TRAP
If we train using 'LEAKED_future', the model is cheating.
Trap neutralized: Leaked data removed.


## 4. Data limits

Limitations of this dataset:

No intent/market context: The data indicates that impressions dropped, but cannot tell us why (e.g., search volume for the topic dropped globally).

Anonymized Queries: Google Search Console hides data for low-volume queries. We may miss long-tail keyword decay.

Directional, not causal: The model provides decision-support by flagging candidates likely to decay. It does not guarantee that refreshing the content will reverse it.

In [6]:
# Data limits confirmed.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.